# Классификация трафика автоматизированного сбора данных

**Мой основной результат на трейне:** на отдельно отложенных датах 17–19 апреля 2026 года
`Precision при Recall ≥ 0.70 = 0.807143`.

Модель: среднее между ансамблем трёх CatBoost на базовых признаках и LightGBM на расширенном наборе признаков.
Конфигурация выбрана по двум более ранним разбиениям, затем зафиксирована до проверки 17–19 апреля.

Ноутбук нужно открыть **в корне распакованного архива**, рядом с `solution.py` и папкой `data`.
Более подробно про запуск файлов я писал в ридми.

In [2]:
! pip install --only-binary=:all: -r requirements.txt

ERROR: Ignored the following yanked versions: 3.0.4
ERROR: Could not find a version that satisfies the requirement pandas==2.2.3 (from versions: 2.3.3, 3.0.0rc0, 3.0.0rc1, 3.0.0rc2, 3.0.0, 3.0.1, 3.0.2, 3.0.3, 3.0.5, 3.0.6)
ERROR: No matching distribution found for pandas==2.2.3


In [3]:
from pathlib import Path
import json
import sys
import importlib.metadata
import numpy as np
import pandas as pd

ROOT = Path.cwd()
assert (ROOT / "solution.py").exists(), "Откройте ноутбук в корне архива решения"
from solution import run, sha256, validate_submission
from metric import precision_at_recall

print("Python:", sys.version.split()[0])
for package in ["numpy", "pandas", "scipy", "scikit-learn", "catboost", "lightgbm"]:
    print(package, importlib.metadata.version(package))

Python: 3.14.7
numpy 2.3.5
pandas 2.3.3
scipy 1.17.0
scikit-learn 1.8.0
catboost 1.2.10
lightgbm 4.6.0


## 1. Постановка и официальная метрика

Единица предсказания — `cookie_id` внутри заданного суточного окна. Положительный класс —
трафик известных сервисов автоматизированного сбора данных, отрицательный — трафик,
отнесённый к человеческому. Это не разметка всех возможных ботов в интернете.

Проверяющая система сама выбирает порог, максимизирующий precision среди порогов с recall не ниже 70%.
Одинаковые значения score нельзя разделять. Используется **оригинальный `metric.py` из выданного архива**,
а не приближение метрики и не значение precision при фиксированном пороге 0.5.

Ниже — два важных пограничных случая: максимум может находиться после первой точки с нужной полнотой;
группа одинаковых score включается целиком.

In [2]:
assert precision_at_recall([1, 0, 1, 1, 1], [5, 4, 3, 2, 1]) == 0.8
assert precision_at_recall([1, 1, 0, 0], [0.5] * 4) == 0.5
print("Проверки определения метрики пройдены")

Проверки определения метрики пройдены


## 2. Данные и очистка

В train 11 091 куки, из них 899 положительных; в test 4 909 куки. Изначально имеется 328 905 событий.
Сначала применяется фильтр `window_start_ts <= event_ts < window_end_ts`, затем удаляются полные дубли:
40 779 событий вне окна и 4 293 дубликата **внутри окна**. Остаётся 283 833 события.
Во всём исходном файле 4 863 дубля: 570 из них уже попали в исключённые события вне окна,
поэтому два исходных числа нельзя просто складывать.

Все куки сохраняются; присоединение метаданных проверяется как `many_to_one`.
Отсутствие координат не считается координатой `(0, 0)`. Числовые пропуски обрабатываются моделями нативно.
Платформы нормализуются по регистру и синонимам. Для последовательностей события сортируются по времени;
порядок внутри одной секунды не интерпретируется как причинный.

Аудит ниже сохранён из первого полного запуска. При повторном обучении он будет рассчитан заново.

In [3]:
audit = json.loads((ROOT / "artifacts/data_audit.json").read_text(encoding="utf-8"))
fields = ["train_rows", "test_rows", "positive_train", "raw_events", "events_at_or_after_end",
          "duplicates_within_window", "clean_events", "cookies_without_in_window_events"]
pd.Series({key: audit[key] for key in fields}, name="Количество").to_frame()

,Количество
train_rows,11091
test_rows,4909
positive_train,899
raw_events,328905
events_at_or_after_end,40779
duplicates_within_window,4293
clean_events,283833
cookies_without_in_window_events,0


## 3. Признаки

| Группа | Примеры | Зачем включена |
|---|---|---|
| Типы действий | Число и доли просмотров, контактов, фото, избранного, входов, капч | Различия сценариев использования |
| Время | Квантили интервалов, CV, регулярность, короткие паузы, сессии | Характер и ритм активности |
| Разнообразие | Уникальные объявления, категории, города, запросы; энтропия и повторяемость | Ширина и концентрация интереса |
| Поиск и навигация | Глубина выдачи, шаги страниц внутри запроса, переходы между действиями | Структура обхода контента |
| Курсор | Покрытие, разброс, перемещения, ковариация, близость точек | Геометрия наблюдаемых взаимодействий |
| Контекст | Возраст куки, платформа, семейство браузера и ОС без версий | Контекст поведения без запоминания технических ID |

`features.py` строит 364 базовых признака; `features_extra.py` добавляет 329.
После удаления констант на обучающих строках CatBoost использует 355 признаков, LightGBM — 665.

Не используются `cookie_id`, сырые значения `item_id`, полные User-Agent, абсолютные даты и события за пределами окна.
Нет кодирования категорий по целевой переменной, псевдоразметки, ручных меток test или статистик, переносящих информацию между куками.
Построение признаков сразу для train и test безопасно здесь, потому что **каждая кука обрабатывается независимо**.

Расширенный набор показал себя слабее, но его ошибки дополняют ошибки CatBoost:
польза проверена на двух временных разбиениях.

## 4. Валидация и выбор решения

| Назначение | Обучение | Проверка | Куки / положительные на проверке |
|---|---|---|---|
| Валидация A | 6–10 апреля | 11–13 апреля | 2 556 / 199 |
| Валидация B | 6–13 апреля | 14–16 апреля | 2 367 / 195 |
| Отложенная проверка | 6–16 апреля | 17–19 апреля | 1 951 / 160 |
| Итоговое обучение | 6–19 апреля | Скрытый test: 20–26 апреля | 4 909 / неизвестно |

Сравнивались двухпризнаковый RandomForest, LightGBM разных размеров и весов класса,
CatBoost, XGBoost, ExtraTrees, расширенные признаки, отбор признаков и усреднение моделей.
Для отбора ансамбля использовалась основная метрика, усреднённая **по двум разбиениям для подбора моделей**,
с AP как дополнительным критерием при равенстве. По сути своей это среднее метрик, а не метрика объединённой выборки.

Широкий подбор способен завысить оценки при подборе моделей; для этого и оставлена отдельная проверка.
Гиперпараметры и веса основного решения после её просмотра не менялись.

Ниже выводятся сохранённые результаты реальных запусков.
Полный журнал экспериментов находится в `experiments` и `crossval`;
скрипты повторения исследовательской части — в `research`.

In [4]:
summary = pd.read_csv(ROOT / "artifacts/validation_summary.csv")
view = summary[summary["model"].isin(["ensemble", "baseline_random_forest"])].copy()
view[["fold", "model", "n_cookies", "n_bots", "precision_at_recall_ge_0_70",
      "average_precision", "roc_auc"]].round(6)

,fold,model,n_cookies,n_bots,precision_at_recall_ge_0_70,average_precision,roc_auc
0,early,ensemble,2556,199,0.838323,0.779662,0.932765
1,early,baseline_random_forest,2556,199,0.093220,0.231783,0.625173
4,late,ensemble,2367,195,0.816568,0.808982,0.947596
5,late,baseline_random_forest,2367,195,0.099639,0.229475,0.620736
8,holdout,ensemble,1951,160,0.807143,0.800142,0.941063
11,holdout,baseline_random_forest,1951,160,0.095718,0.163416,0.572405


### Отложенная проверка

Precision = **113 / 140 = 80.71%**, recall = **113 / 160 = 70.625%**.
27 отрицательных куки были отмечены как положительные при диагностическом оптимальном пороге.
AP = 0.800142; ROC-AUC = 0.941063. AP здесь означает `average_precision_score`.

Одиночный LightGBM на этой отложенной выборке получил 0.829630, больше ансамбля, но на обоих разбиениях для подбора моделей
был с существенно более низким скором (0.769231 и 0.744565). Основной вариант не заменял на него после просмотра результатов отложенной проверки:
иначе эта выборка превратилась бы в ещё одну выборку подбора.

Для независимого повторного расчёта на отложенной выборке можно включить следующую опцию. По умолчанию
ноутбук не тратит время на повторную валидацию, но **полностью переобучает итоговые модели ниже**.

In [5]:
RECALCULATE_HOLDOUT = False
if RECALCULATE_HOLDOUT:
    run(ROOT / "data", ROOT / "holdout_recheck", ROOT / "final_config.json", mode="validate")
else:
    print("Показаны сохранённые результаты holdout; для пересчёта установите True.")

Показаны сохранённые результаты holdout; для пересчёта установите True.


## 5. Финальный ансамбль

`score = 0.5 × mean(CatBoost_seed42, CatBoost_seed1337, CatBoost_seed2026) + 0.5 × LightGBM`.

У CatBoost 1 600 деревьев глубины 6, скорость обучения 0.035; у LightGBM 300 деревьев,
15 листьев, скорость обучения 0.03. Обе модели обучаются с логарифмической функцией потерь;
конфигурации выбирались по требуемой precision-метрике.
Полный набор параметров задан в `solution.py`, веса и семена — в `final_config.json`.

Усреднение выполняется для каждой куки отдельно. Ранговые преобразования по всему test,
калибровка на test, изменение порога и округление score не применяются.

In [6]:
config = json.loads((ROOT / "final_config.json").read_text(encoding="utf-8"))
print(json.dumps(config["members"], indent=2, ensure_ascii=False))
print("Средняя development-метрика:", config["selection"]["mean_precision_at_recall70"])
print("Holdout использован для подбора:", config["selection"]["holdout_used_for_selection"])

[
  {
    "family": "catboost",
    "features": "base",
    "iterations": 1600,
    "depth": 6,
    "seeds": [
      42,
      1337,
      2026
    ],
    "weight": 0.5
  },
  {
    "family": "lightgbm",
    "features": "augmented",
    "iterations": 300,
    "num_leaves": 15,
    "seeds": [
      42
    ],
    "weight": 0.5
  }
]
Средняя development-метрика: 0.8274457003153456
Holdout использован для подбора: False


## 6. Полное повторное обучение и создание файла предсказаний

Следующая ячейка заново читает исходные CSV, строит признаки, обучает модели **на всех 11 091 размеченных куки**
и создаёт `reproduced/submission.csv`. Готовые веса и готовые предсказания в обучении не используются.
В конце также проверяется совпадение предсказаний после сохранения и загрузки моделей.

Запуск занимает несколько минут на CPU. Модели и CSV из исходного решения лежат в `artifacts`;
отдельная папка `reproduced` нужна, чтобы не перезаписывает эталон.

In [7]:
OUTPUT = ROOT / "reproduced"
run(ROOT / "data", OUTPUT, ROOT / "final_config.json", mode="train")

{
  "train_rows": 11091,
  "test_rows": 4909,
  "raw_events": 328905,
  "positive_train": 899,
  "exact_duplicate_rows": 4863,
  "events_before_window": 0,
  "events_at_or_after_end": 40779,
  "events_within_window_before_dedup": 288126,
  "duplicates_within_window": 4293,
  "clean_events": 283833,
  "cookies_without_in_window_events": 0,
  "events_before_cookie_creation": 0,
  "feature_count": 364,
  "categorical_features": [
    "platform_norm_mode",
    "browser_mode",
    "os_mode"
  ],
  "feature_build_seconds": 28.8,
  "train_date_min": "2026-04-06 00:00:00",
  "train_date_max": "2026-04-19 00:00:00",
  "test_date_min": "2026-04-20 00:00:00",
  "test_date_max": "2026-04-26 00:00:00"
}


Extra features: (16000, 329) elapsed 25.71


catboost, seed=42, trees=1600, features=355, seconds=36.0


catboost, seed=1337, trees=1600, features=355, seconds=36.3


catboost, seed=2026, trees=1600, features=355, seconds=35.8


lightgbm, seed=42, trees=300, features=665, seconds=4.1


Saved /mnt/data/antibot_solution/reproduced/submission.csv (4909 rows), SHA-256: 7e85915d437f9ba0795158ab6125f5ff34a800667f6abb5925af665fa4dc2f7e


## 7. Проверка формата и совпадения хэшей

Сравнение с эталонным CSV выполняется **только после независимого обучения** и не влияет на предсказания.
Проверяются две колонки, все test-куки в исходном порядке, отсутствие дублей/NaN,
диапазон [0, 1], точные числовые значения и SHA-256 файла.
Если при вашем совпадении хэш отличается, то мб у вас другая версия библиотеки или установленного питона. Оно может влиять на отличия.

In [8]:
test = pd.read_csv(ROOT / "data/test.csv")
submission = pd.read_csv(OUTPUT / "submission.csv", float_precision="round_trip")
reference = pd.read_csv(ROOT / "submission.csv", float_precision="round_trip")
validate_submission(submission, test)
assert submission.cookie_id.tolist() == reference.cookie_id.tolist()
assert np.array_equal(submission.score.to_numpy(), reference.score.to_numpy())
assert sha256(OUTPUT / "submission.csv") == sha256(ROOT / "submission.csv")
print("Куки:", len(submission))
print("SHA-256:", sha256(OUTPUT / "submission.csv"))
print("CSV воспроизведён побитно.")
submission.head()

Куки: 4909
SHA-256: 7e85915d437f9ba0795158ab6125f5ff34a800667f6abb5925af665fa4dc2f7e
CSV воспроизведён побитно.


,cookie_id,score
0,ck_315fb710a0e371e7,0.000881
1,ck_a76ee3b3e3e522fd,0.028577
2,ck_94c9a4d382689e82,0.088234
3,ck_8eaf9509ad9462a0,0.000879
4,ck_9a88a5a989cb5bc6,0.007449


In [9]:
import subprocess
result = subprocess.run([sys.executable, str(ROOT / "tests.py")], check=True,
                        capture_output=True, text=True, cwd=ROOT)
print(result.stdout + result.stderr)

test_boundaries_duplicates_and_grain (__main__.FeatureTests.test_boundaries_duplicates_and_grain) ... ok
test_cookie_identifiers_do_not_affect_features (__main__.FeatureTests.test_cookie_identifiers_do_not_affect_features) ... ok
test_invariance_to_future_events_and_labels (__main__.FeatureTests.test_invariance_to_future_events_and_labels) ... ok
test_against_independent_brute_force (__main__.MetricTests.test_against_independent_brute_force) ... ok
test_official_examples_and_ties (__main__.MetricTests.test_official_examples_and_ties) ... ok
test_reject_invalid_rows_or_scores (__main__.SubmissionTests.test_reject_invalid_rows_or_scores) ... ok
test_valid_format (__main__.SubmissionTests.test_valid_format) ... ok

----------------------------------------------------------------------
Ran 7 tests in 0.369s

OK

